In [1]:
# pip uninstall apex

In [2]:
import os
import yaml
import random
import torch
import evaluate
from dotenv import load_dotenv
from datasets import load_dataset, DatasetDict, Audio
from huggingface_hub import notebook_login
from transformers import (
    WhisperFeatureExtractor, 
    WhisperTokenizer, 
    WhisperProcessor, 
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from dataclasses import dataclass
from typing import *


if os.path.exists(".env"):
    load_dotenv()
else:
    load_dotenv("env.env")

WHISPER_MODEL = "openai/whisper-medium"
MODEL_OUTPUT_DIR = "model-" + WHISPER_MODEL.replace("/", "-")
LOGS_OUTPUT_DIR = "logs-" + WHISPER_MODEL.replace("/", "-")
LANGUAGE = "Arabic"
LANGUAGE_TAG = "ar"
TASK = "transcribe"
EVALUATION_METRIC = "wer"
AUDIO_SAMPLE_RATE = 16000
CONFIG_PATH = "./config.yaml"
AUTH_TOKEN = os.getenv("HUGGINGFACE_ACCESS_TOKEN")

# notebook_login()

2024-05-09 18:10:32.115820: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-05-09 18:10:32.115893: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-05-09 18:10:32.123337: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-05-09 18:10:32.146611: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-05-09 18:10:33.145847: W tensorflow/compiler/tf2

[2024-05-09 18:10:39,942] [INFO] [real_accelerator.py:158:get_accelerator] Setting ds_accelerator to cuda (auto detect)


In [3]:

common_voice = DatasetDict()

common_voice["train"] = load_dataset(
    "mozilla-foundation/common_voice_11_0", 
    LANGUAGE_TAG, 
    split="train+validation", 
    use_auth_token=AUTH_TOKEN
)

common_voice["test"] = load_dataset(
    "mozilla-foundation/common_voice_11_0", 
    LANGUAGE_TAG, 
    split="test", 
    use_auth_token=AUTH_TOKEN
)

print(common_voice)


Found cached dataset common_voice_11_0 (/root/.cache/huggingface/datasets/mozilla-foundation___common_voice_11_0/ab/11.0.0/3f27acf10f303eac5b6fbbbe02495aeddb46ecffdb0a2fe3507fcfbf89094631)
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1421: FutureWarning: promote has been superseded by promote_options='default'.
  table = cls._concat_blocks(blocks, axis=0)
Found cached dataset common_voice_11_0 (/root/.cache/huggingface/datasets/mozilla-foundation___common_voice_11_0/ab/11.0.0/3f27acf10f303eac5b6fbbbe02495aeddb46ecffdb0a2fe3507fcfbf89094631)


DatasetDict({
    train: Dataset({
        features: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment'],
        num_rows: 30179
    })
    test: Dataset({
        features: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment'],
        num_rows: 9124
    })
})


In [4]:
common_voice = common_voice.remove_columns(
    [
        "accent", 
        "age", 
        "client_id", 
        "down_votes", 
        "gender", 
        "locale", 
        "path", 
        "segment", 
        "up_votes"
    ]
)

common_voice

DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 30179
    })
    test: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 9124
    })
})

In [5]:
common_voice["train"][0]

{'audio': {'path': '/root/.cache/huggingface/datasets/downloads/extracted/cd846e8fc8387ba609d18c0dd7087bbd257d8ed5cada8dec71618daa6849263d/ab_train_0/common_voice_ab_27190037.mp3',
  'array': array([-1.77635684e-14, -2.13162821e-14, -7.10542736e-15, ...,
          7.78658141e-05,  2.44756986e-04,  1.61686330e-04]),
  'sampling_rate': 48000},
 'sentence': 'Уи иагьазхаҳҵеит. Уажәшьҭа ҳанцәа зынӡа аваҟәыла диваҳаргылаӡом.'}

In [6]:
feature_extractor = WhisperFeatureExtractor.from_pretrained(WHISPER_MODEL)
tokenizer = WhisperTokenizer.from_pretrained(WHISPER_MODEL, language=LANGUAGE, task=TASK)

feature_extractor, tokenizer

(WhisperFeatureExtractor {
   "chunk_length": 30,
   "feature_extractor_type": "WhisperFeatureExtractor",
   "feature_size": 80,
   "hop_length": 160,
   "n_fft": 400,
   "n_samples": 480000,
   "nb_max_frames": 3000,
   "padding_side": "right",
   "padding_value": 0.0,
   "processor_class": "WhisperProcessor",
   "return_attention_mask": false,
   "sampling_rate": 16000
 },
 WhisperTokenizer(name_or_path='openai/whisper-small', vocab_size=50258, model_max_length=1024, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'bos_token': AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False), 'eos_token': AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False), 'unk_token': AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False), 'pad_token': AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False), 'additional_special_

In [7]:
sample_idx = random.randint(0, len(common_voice["train"]))
input_sentence = common_voice["train"][sample_idx]["sentence"]
labels = tokenizer(input_sentence).input_ids
decoded_with_special = tokenizer.decode(labels, skip_special_tokens=False)
decoded_str = tokenizer.decode(labels, skip_special_tokens=True)

print(f"Input:                 {input_sentence}")
print(f"Decoded w/ special:    {decoded_with_special}")
print(f"Decoded w/out special: {decoded_str}")
print(f"Are equal:             {input_sentence == decoded_str}")

Input:                 Ихылан иӡсогьы иазааигәан инхо ркәатақәа ракәӡоуп.
Decoded w/ special:    <|startoftranscript|><|ar|><|transcribe|><|notimestamps|>Ихылан иӡсогьы иазааигәан инхо ркәатақәа ракәӡоуп.<|endoftext|>
Decoded w/out special: Ихылан иӡсогьы иазааигәан инхо ркәатақәа ракәӡоуп.
Are equal:             True


In [8]:
processor = WhisperProcessor.from_pretrained(WHISPER_MODEL, language=LANGUAGE, task=TASK)
processor

WhisperProcessor:
- feature_extractor: WhisperFeatureExtractor {
  "chunk_length": 30,
  "feature_extractor_type": "WhisperFeatureExtractor",
  "feature_size": 80,
  "hop_length": 160,
  "n_fft": 400,
  "n_samples": 480000,
  "nb_max_frames": 3000,
  "padding_side": "right",
  "padding_value": 0.0,
  "processor_class": "WhisperProcessor",
  "return_attention_mask": false,
  "sampling_rate": 16000
}

- tokenizer: WhisperTokenizer(name_or_path='openai/whisper-small', vocab_size=50258, model_max_length=1024, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'bos_token': AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False), 'eos_token': AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False), 'unk_token': AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False), 'pad_token': AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, no

In [9]:
common_voice = common_voice.cast_column("audio", Audio(sampling_rate=AUDIO_SAMPLE_RATE))

common_voice["train"][0]

{'audio': {'path': '/root/.cache/huggingface/datasets/downloads/extracted/cd846e8fc8387ba609d18c0dd7087bbd257d8ed5cada8dec71618daa6849263d/ab_train_0/common_voice_ab_27190037.mp3',
  'array': array([ 1.45519152e-11, -1.16415322e-10, -4.36557457e-11, ...,
         -5.48839162e-05, -6.36292098e-05,  5.26811637e-05]),
  'sampling_rate': 16000},
 'sentence': 'Уи иагьазхаҳҵеит. Уажәшьҭа ҳанцәа зынӡа аваҟәыла диваҳаргылаӡом.'}

In [10]:
def prepare_dataset(batch, feature_extractor: WhisperFeatureExtractor, tokenizer: WhisperTokenizer):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch


common_voice = common_voice.map(
    prepare_dataset,
    fn_kwargs={"feature_extractor":feature_extractor, "tokenizer":tokenizer},
    remove_columns=common_voice.column_names["train"], 
    num_proc=os.cpu_count()
)

Map (num_proc=8):   0%|          | 0/30179 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/datasets/table.py:1387: FutureWarning: promote has been superseded by promote_options='default'.
  return cls._concat_blocks(pa_tables_to_concat_vertically, axis=0)
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1387: FutureWarning: promote has been superseded by promote_options='default'.
  return cls._concat_blocks(pa_tables_to_concat_vertically, axis=0)
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1387: FutureWarning: promote has been superseded by promote_options='default'.
  return cls._concat_blocks(pa_tables_to_concat_vertically, axis=0)
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1387: FutureWarning: promote has been superseded by promote_options='default'.
  return cls._concat_blocks(pa_tables_to_concat_vertically, axis=0)
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1387: FutureWarning: promote has been superseded by promote_options='default'.
  return cls._concat_blocks(pa_tables_to_co

Map (num_proc=8):   0%|          | 0/9124 [00:00<?, ? examples/s]

In [11]:
model = WhisperForConditionalGeneration.from_pretrained(WHISPER_MODEL)

model.generation_config.language = LANGUAGE
model.generation_config.task = TASK
model.generation_config.forced_decoder_ids = None

model

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (f

In [12]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch


In [13]:
def compute_metrics(pred, metric, metric_label: str):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    metric_val = 100 * metric.compute(predictions=pred_str, references=label_str)
    
    return {metric_label: metric_val}

In [14]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

In [15]:
with open(CONFIG_PATH, "r") as f:
    training_args_config = yaml.safe_load(f)
f.close()

training_args = Seq2SeqTrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    metric_for_best_model=EVALUATION_METRIC,
    logging_dir=LOGS_OUTPUT_DIR,
    **training_args_config
)

In [16]:
metric = evaluate.load(EVALUATION_METRIC)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["test"],
    data_collator=data_collator,
    compute_metrics=lambda pred : compute_metrics(pred, metric, EVALUATION_METRIC),
    tokenizer=processor.feature_extractor,
)
trainer.train()

/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
`use_cache = True` is incompatible with gradient checkpointing. Setting `use_cache = False`...


Step,Training Loss,Validation Loss,Wer
1000,0.360000,0.402025,68.667579
2000,0.256400,0.314359,57.821183
3000,0.233100,0.271995,51.202621
4000,0.146800,0.253872,50.694617
5000,0.143900,0.246798,49.021315


/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:429: UserWarning: torch.

TrainOutput(global_step=5000, training_loss=0.3648974614620209, metrics={'train_runtime': 16589.166, 'train_samples_per_second': 2.411, 'train_steps_per_second': 0.301, 'total_flos': 1.15419730857984e+19, 'train_loss': 0.3648974614620209, 'epoch': 1.33})